# Bayesian Network Weather Forecasting
Uses the cleaned GSOD 2002 data to build and train a Bayesian Network.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.estimators import MaximumLikelihoodEstimator, BayesianEstimator
from pgmpy.inference import VariableElimination
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

df = pd.read_csv('../data/processed/gsod_2002.csv', low_memory=False)
print(f'Loaded {len(df):,} records with {df.shape[1]} columns')
df.head()

## Discretize Continuous Variables
Bayesian Networks need discrete states. This is going to bin each continuous variable into weather categories.

In [ ]:
disc = pd.DataFrame()

# TEMP (°F): cold < 32, warm > 75, else avg
disc['TEMP_D'] = pd.cut(
    df['TEMP'],
    bins=[-np.inf, 32, 75, np.inf],
    labels=['cold', 'avg', 'warm']
)

# SLP: sea-level pressure (mbar): low < 1005, high >= 1005
disc['SLP_D'] = pd.cut(
    df['SLP'],
    bins=[-np.inf, 1005, np.inf],
    labels=['low', 'high']
)

# WDSP: mean wind speed (knots): calm < 5, moderate < 15, strong >= 15
disc['WIND_D'] = pd.cut(
    df['WDSP'],
    bins=[-np.inf, 5, 15, np.inf],
    labels=['calm', 'moderate', 'strong']
)

# DEWP: dew point (°F): low < 32, moderate < 55, high >= 55
disc['DEWP_D'] = pd.cut(
    df['DEWP'],
    bins=[-np.inf, 32, 55, np.inf],
    labels=['low', 'moderate', 'high']
)

# PRCP: precipitation (inches): none = 0, light < 0.5, heavy >= 0.5
disc['PRCP_D'] = pd.cut(
    df['PRCP'].fillna(0),
    bins=[-np.inf, 0.0, 0.5, np.inf],
    labels=['none', 'light', 'heavy']
)

# Binary event flags
disc['FOG']     = df['FOG'].astype(int)
disc['RAIN']    = df['RAIN'].astype(int)
disc['SNOW']    = df['SNOW'].astype(int)
disc['THUNDER'] = df['THUNDER'].astype(int)

# Drop rows with any remaining NaN (this is from continuous sentinels)
disc.dropna(inplace=True)

# Convert categoric stuff to string so pgmpy can consume them cleanly
for col in ['TEMP_D', 'SLP_D', 'WIND_D', 'DEWP_D', 'PRCP_D']:
    disc[col] = disc[col].astype(str)

print(f'Discretized dataset: {len(disc):,} rows')
print(disc.dtypes)
disc.head()

In [ ]:
# distribution check of discretized variables
fig, axes = plt.subplots(3, 3, figsize=(14, 10))
axes = axes.flatten()

for ax, col in zip(axes, disc.columns):
    disc[col].value_counts().sort_index().plot(kind='bar', ax=ax)
    ax.set_title(col)
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=30)

for ax in axes[len(disc.columns):]:
    ax.set_visible(False)

plt.suptitle('Discretized Variable Distributions', fontsize=14)
plt.tight_layout()
plt.show()

## Network Structure (the DAG)

Edge reasoning:
- **SLP -> TEMP**: pressure systems effect temperature patterns
- **SLP -> WIND**: pressure gradient creates wind
- **SLP -> PRCP**: low pressure brings precipitation
- **TEMP -> DEWP**: temperature controls dewpoint proximity
- **TEMP -> PRCP**: temperature determines rain vs snow
- **TEMP -> SNOW**: cold required for snow
- **DEWP -> FOG**: high dew point near temp -> fog
- **DEWP -> RAIN**: moisture content makes rain
- **PRCP -> RAIN**: precipitation type
- **PRCP -> SNOW**: precipitation type
- **WIND -> FOG**: wind disperses fog
- **TEMP -> THUNDER**: instability of warmth + moisture = thunder
- **DEWP -> THUNDER**: humidity helps thunderstorm development

In [ ]:
edges = [
    ('SLP_D',  'TEMP_D'),
    ('SLP_D',  'WIND_D'),
    ('SLP_D',  'PRCP_D'),
    ('TEMP_D', 'DEWP_D'),
    ('TEMP_D', 'PRCP_D'),
    ('TEMP_D', 'SNOW'),
    ('TEMP_D', 'THUNDER'),
    ('DEWP_D', 'FOG'),
    ('DEWP_D', 'RAIN'),
    ('DEWP_D', 'THUNDER'),
    ('PRCP_D', 'RAIN'),
    ('PRCP_D', 'SNOW'),
    ('WIND_D', 'FOG'),
]

model = DiscreteBayesianNetwork(edges)
print('Nodes:', model.nodes())
print('Edges:', model.edges())

## Train / Test Split and Parameter Learning here

In [ ]:
train_df, test_df = train_test_split(disc, test_size=0.2, random_state=42)
print(f'Train: {len(train_df):,}  |  Test: {len(test_df):,}')

# Bayesian estimator with K2 (Dirichlet) prior to handle zero count cells
model.fit(
    train_df,
    estimator=BayesianEstimator,
    prior_type='K2'
)

print('\nModel trained successfully.')
print('CPD nodes:', [cpd.variable for cpd in model.cpds])

In [ ]:
# Inspecting rain, snow ,fog
for var in ['RAIN', 'SNOW', 'FOG']:
    cpd = model.get_cpds(var)
    print(cpd)
    print()

## Checking model here
Using variable elimination here

In [ ]:
infer = VariableElimination(model)

scenarios = [
    {'label': 'Cold + Low Pressure',    'evidence': {'TEMP_D': 'cold', 'SLP_D': 'low'}},
    {'label': 'Warm + High Pressure',   'evidence': {'TEMP_D': 'warm', 'SLP_D': 'high'}},
    {'label': 'Avg Temp + Strong Wind', 'evidence': {'TEMP_D': 'avg',  'WIND_D': 'strong'}},
    {'label': 'High Dew + Low Pressure','evidence': {'DEWP_D': 'high', 'SLP_D': 'low'}},
]

query_vars = ['RAIN', 'SNOW', 'FOG', 'THUNDER']

for scenario in scenarios:
    print(f"=== {scenario['label']} ===")
    for var in query_vars:
        result = infer.query(variables=[var], evidence=scenario['evidence'], show_progress=False)
        p = result.values
        labels = result.state_names[var]
        probs = {str(l): round(float(v), 4) for l, v in zip(labels, p)}
        print(f'  {var}: {probs}')
    print()

## Evaluation
Predicting and then comparing against actual

In [ ]:
# Evidence variables so everything except what we are predicting
EVIDENCE_COLS = ['SLP_D', 'TEMP_D', 'WIND_D', 'DEWP_D', 'PRCP_D']
TARGET_COLS   = ['RAIN', 'SNOW', 'FOG']

# Using a sample for speed if the test set is large
eval_df = test_df.sample(min(2000, len(test_df)), random_state=0).reset_index(drop=True)

predictions = {t: [] for t in TARGET_COLS}

for _, row in eval_df.iterrows():
    evidence = {col: row[col] for col in EVIDENCE_COLS if pd.notna(row[col])}
    for target in TARGET_COLS:
        try:
            q = infer.map_query(variables=[target], evidence=evidence, show_progress=False)
            predictions[target].append(q[target])
        except Exception:
            predictions[target].append(np.nan)

print('Evaluation complete.')

In [ ]:
for target in TARGET_COLS:
    pred = pd.array(predictions[target])
    actual = eval_df[target].values

    # Drop rows where prediction failed
    mask = pd.notna(pred)
    pred   = pred[mask].astype(int)
    actual = actual[mask].astype(int)

    print(f'=== {target} ===')
    print(classification_report(actual, pred, zero_division=0))

    cm = confusion_matrix(actual, pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=[f'pred_{i}' for i in range(cm.shape[1])],
                yticklabels=[f'true_{i}' for i in range(cm.shape[0])])
    plt.title(f'Confusion Matrix – {target}')
    plt.tight_layout()
    plt.show()
    print()

### Static BN Notes

The static BN establishes a same day conditional inference baseline using a single year (2002) of globally aggregated GSOD data. The sections below extend this to a **Dynamic Bayesian Network** for next day forecasting and then to a **Latent State DBN** with HMM inferred hidden atmospheric regimes. This si to compare more dynamic modela against the basic bayesian network

## Dynamic Bayesian Network (DBN): Temporal Extension

A standard static BN can only explain *current* conditions. It cannot forecast *future* ones. To enable next day prediction we extend to a **2 Time Slice BN (2 TBN)**.

The idea: every row in the dataset becomes a `(yesterday -> today)` transition pair. Yesterdays discretised variables become `_prev` root nodes. Todays variables are the targets. Two classes of edges are added here to make it work:

- **Inter slice edges** (`_prev -> current`): capture temporal persistence (cold air masses linger for example) and lagged atmospheric influence (think yesterday's pressure impacts todayss temperature).
- **Intra slice edges** (same as the static BN): capture same day physical causality within the current time step.

At inference time we provide "todays" observations as evidence for the `_prev` nodes and query the model for tomorrow's weather events (`RAIN`, `SNOW`, `FOG`, `THUNDER`).

In [ ]:
# ── DBN Data Preparation ──────────────────────────────────────────────────────
# Build consecutive day transition pairs per station.
# Each row = (yesterdays discretised vars) paired with (todays vars)

df_src = pd.read_csv('../data/processed/gsod_2002.csv', low_memory=False)

_cut_specs = {
    'TEMP_D': (df_src['TEMP'],            [-np.inf, 32, 75, np.inf],    ['cold', 'avg', 'warm']),
    'SLP_D':  (df_src['SLP'],             [-np.inf, 1005, np.inf],       ['low', 'high']),
    'WIND_D': (df_src['WDSP'],            [-np.inf, 5, 15, np.inf],      ['calm', 'moderate', 'strong']),
    'DEWP_D': (df_src['DEWP'],            [-np.inf, 32, 55, np.inf],     ['low', 'moderate', 'high']),
    'PRCP_D': (df_src['PRCP'].fillna(0),  [-np.inf, 0.0, 0.5, np.inf],  ['none', 'light', 'heavy']),
}

dbn_src = pd.DataFrame()
for col, (series, bins, labels) in _cut_specs.items():
    dbn_src[col] = pd.cut(series, bins=bins, labels=labels).astype(str)

for flag in ['FOG', 'RAIN', 'SNOW', 'THUNDER']:
    dbn_src[flag] = df_src[flag].astype(int)

dbn_src['STN']  = df_src['STN'].astype(str)
dbn_src['DATE'] = pd.to_datetime(df_src['DATE'])

# Drop rows where any continuous discretised columns are missing
dbn_src.dropna(subset=list(_cut_specs.keys()), inplace=True)
dbn_src = dbn_src.sort_values(['STN', 'DATE']).reset_index(drop=True)

# delay every variable by one day, grouped per station to create transition pairs
# groupby().shift(1) makes the first record of each station produce NaN
# preventing cross station contamination at year boundaries
SHIFT_COLS = list(_cut_specs.keys()) + ['FOG', 'RAIN', 'SNOW', 'THUNDER']
for col in SHIFT_COLS:
    dbn_src[f'{col}_prev'] = dbn_src.groupby('STN')[col].shift(1)

dbn_df = dbn_src.dropna(subset=[f'{c}_prev' for c in SHIFT_COLS]).copy()

for col in ['FOG', 'RAIN', 'SNOW', 'THUNDER']:
    dbn_df[f'{col}_prev'] = dbn_df[f'{col}_prev'].astype(int)

# Drop STN keep DATE for the temporal train/test split coming up
dbn_df = dbn_df.drop(columns=['STN'])

print(f'DBN transition rows: {len(dbn_df):,}   columns: {dbn_df.shape[1]}')
print(dbn_df.dtypes)
dbn_df.head()

### DBN Structure 2 Time Slice DAG

**Inter-slice edges** (t-1 -> t): temporal persistence for all 5 atmospheric variables, two cross-variable lagged influences, and persistence for each binary weather event.

**Intra-slice edges** (t -> t): identical to the static BN: same day physical causality is preserved unchanged

In [ ]:
# ── DBN Structure ─────────────────────────────────────────────────────────────
# All _prev nodes are root nodes so no parents in this 2 TBN.
# The full graph stays acyclic: prev -> current only, never the reverse

inter_slice_edges = [
    # Atmospheric persistence so each variable carries forward one day
    ('TEMP_D_prev', 'TEMP_D'),
    ('SLP_D_prev',  'SLP_D'),
    ('WIND_D_prev', 'WIND_D'),
    ('DEWP_D_prev', 'DEWP_D'),
    ('PRCP_D_prev', 'PRCP_D'),
    # Cross variable temporal influence
    ('SLP_D_prev',  'TEMP_D'),   # yesterdays pressure system influences todays temperature
    ('TEMP_D_prev', 'DEWP_D'),   # yesterdays temperature influences todays moisture
    # Binary weather event persistence (wet/foggy/stormy streaks are common hence the streak)
    ('RAIN_prev',    'RAIN'),
    ('SNOW_prev',    'SNOW'),
    ('FOG_prev',     'FOG'),
    ('THUNDER_prev', 'THUNDER'),
]

intra_slice_edges = [
    # Same atmospheric causality as the static BN
    ('SLP_D',  'TEMP_D'),
    ('SLP_D',  'WIND_D'),
    ('SLP_D',  'PRCP_D'),
    ('TEMP_D', 'DEWP_D'),
    ('TEMP_D', 'PRCP_D'),
    ('TEMP_D', 'SNOW'),
    ('TEMP_D', 'THUNDER'),
    ('DEWP_D', 'FOG'),
    ('DEWP_D', 'RAIN'),
    ('DEWP_D', 'THUNDER'),
    ('PRCP_D', 'RAIN'),
    ('PRCP_D', 'SNOW'),
    ('WIND_D', 'FOG'),
]

dbn_model = DiscreteBayesianNetwork(inter_slice_edges + intra_slice_edges)

print(f'Nodes ({len(dbn_model.nodes())}):')
print('  prev (roots):', sorted(n for n in dbn_model.nodes() if '_prev' in n))
print('  current:     ', sorted(n for n in dbn_model.nodes() if '_prev' not in n))
print(f'\nEdges ({len(dbn_model.edges())}):')
for src, dst in sorted(dbn_model.edges()):
    print(f'  {src:20s} → {dst}')

In [ ]:
# ── Time Ordered Train / Test Split + DBN Parameter Learning ──────────────────
# Split by date rather than randomly to prevent look ahead bias so..
# the model never sees future observations during training

dbn_sorted = dbn_df.sort_values('DATE').reset_index(drop=True)
split_idx  = int(len(dbn_sorted) * 0.8)
split_date = dbn_sorted.iloc[split_idx]['DATE']

dbn_train = dbn_sorted.iloc[:split_idx].drop(columns=['DATE'])
dbn_test  = dbn_sorted.iloc[split_idx:].drop(columns=['DATE'])

print(f'Train: {len(dbn_train):,} rows  (up to   {split_date.date()})')
print(f'Test:  {len(dbn_test):,}  rows  (after   {split_date.date()})')

# Bayesian Estimator with K2 (Dirichlet) prior so this handles zero count CPT cells
dbn_model.fit(dbn_train, estimator=BayesianEstimator, prior_type='K2')

print('\nDBN trained successfully.')
print('CPDs learned for:', [cpd.variable for cpd in dbn_model.cpds])

## DBN Forecasting: Next-Day Prediction

Evidence = todays full observation (supplied as `_prev` variables)
Query = tomorrows weather events (`RAIN`, `SNOW`, `FOG`, `THUNDER`)

This is an actual forecast having not seeing anything before

In [ ]:
# ── Next Day Forecast Scenarios ───────────────────────────────────────────────
# Supply todays observations as _prev evidence and query tomorrows events

dbn_infer = VariableElimination(dbn_model)

dbn_scenarios = [
    {
        'label': 'Warm + Low Pressure + Light Rain (approaching storm)',
        'evidence': {
            'TEMP_D_prev': 'warm',  'SLP_D_prev':  'low',
            'WIND_D_prev': 'moderate', 'DEWP_D_prev': 'high',
            'PRCP_D_prev': 'light', 'RAIN_prev': 1,
            'SNOW_prev': 0, 'FOG_prev': 0, 'THUNDER_prev': 0,
        }
    },
    {
        'label': 'Cold + High Pressure + No Precipitation (clear winter day)',
        'evidence': {
            'TEMP_D_prev': 'cold',  'SLP_D_prev':  'high',
            'WIND_D_prev': 'calm',  'DEWP_D_prev': 'low',
            'PRCP_D_prev': 'none',  'RAIN_prev': 0,
            'SNOW_prev': 0, 'FOG_prev': 0, 'THUNDER_prev': 0,
        }
    },
    {
        'label': 'Avg Temp + Low Pressure + Strong Wind + Heavy Rain',
        'evidence': {
            'TEMP_D_prev': 'avg',   'SLP_D_prev':  'low',
            'WIND_D_prev': 'strong','DEWP_D_prev': 'high',
            'PRCP_D_prev': 'heavy', 'RAIN_prev': 1,
            'SNOW_prev': 0, 'FOG_prev': 0, 'THUNDER_prev': 1,
        }
    },
    {
        'label': 'Warm + High Pressure + Calm Wind + High Dew (fog candidate)',
        'evidence': {
            'TEMP_D_prev': 'warm',  'SLP_D_prev':  'high',
            'WIND_D_prev': 'calm',  'DEWP_D_prev': 'high',
            'PRCP_D_prev': 'none',  'RAIN_prev': 0,
            'SNOW_prev': 0, 'FOG_prev': 1, 'THUNDER_prev': 0,
        }
    },
]

QUERY_VARS = ['RAIN', 'SNOW', 'FOG', 'THUNDER']

for scenario in dbn_scenarios:
    print(f"=== {scenario['label']} ===")
    for var in QUERY_VARS:
        result = dbn_infer.query(
            variables=[var], evidence=scenario['evidence'], show_progress=False
        )
        probs = {
            str(l): round(float(v), 4)
            for l, v in zip(result.state_names[var], result.values)
        }
        print(f'  P({var}) = {probs}')
    print()

## DBN Evaluation: Next Day Forecast Accuracy

For each row in the test set, the 9 `_prev` columns (yesterdays full observation) are used as evidence, and the model predicts todays `RAIN`, `SNOW`, `FOG`, and `THUNDER` via MAP query. Results are compared against the actual values

In [ ]:
# ── DBN Evaluation: MAP prediction on hold out test set ───────────────────────
# Evidence: all 9 _prev columns (yesterdays complete observation)
# Targets:  RAIN, SNOW, FOG, THUNDER (todays events so genuine next day forecast)

DBN_EVIDENCE_COLS = [
    'TEMP_D_prev', 'SLP_D_prev', 'WIND_D_prev', 'DEWP_D_prev', 'PRCP_D_prev',
    'RAIN_prev', 'SNOW_prev', 'FOG_prev', 'THUNDER_prev',
]
DBN_TARGET_COLS = ['RAIN', 'SNOW', 'FOG', 'THUNDER']

# Sample for speed, use a fixed seed for reproducibility
dbn_eval = dbn_test.sample(min(2000, len(dbn_test)), random_state=0).reset_index(drop=True)

dbn_preds = {t: [] for t in DBN_TARGET_COLS}

for _, row in dbn_eval.iterrows():
    evidence = {col: row[col] for col in DBN_EVIDENCE_COLS if pd.notna(row[col])}
    for target in DBN_TARGET_COLS:
        try:
            q = dbn_infer.map_query(
                variables=[target], evidence=evidence, show_progress=False
            )
            dbn_preds[target].append(q[target])
        except Exception:
            dbn_preds[target].append(np.nan)

print('DBN next day forecast evaluation complete.')

In [ ]:
# ── DBN Metrics & Confusion Matrices ──────────────────────────────────────────
for target in DBN_TARGET_COLS:
    pred   = pd.array(dbn_preds[target])
    actual = dbn_eval[target].values

    mask   = pd.notna(pred)
    pred   = pred[mask].astype(int)
    actual = actual[mask].astype(int)

    print(f'=== {target} (DBN next day forecast) ===')
    print(classification_report(actual, pred, zero_division=0))

    cm = confusion_matrix(actual, pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
                xticklabels=[f'pred_{i}' for i in range(cm.shape[1])],
                yticklabels=[f'true_{i}' for i in range(cm.shape[0])])
    plt.title(f'DBN Confusion Matrix — {target}  (next day forecast)')
    plt.tight_layout()
    plt.show()
    print()

## Naive Persistence Baseline

A **persistence forecast** predicts that tomorrows weather matches todays. This is the canonical sanity check for any weather model so if we cant beat it, the model isnt learning anything. The baseline is evaluated on the same held out test set and the same 2000 row sample used for the DBN, enabling a direct comparison.

In [ ]:

# ── Naive persistence baseline on the DBN hold out set ────────────────────────
# The persistence forecast predicts todays event = yesterdays event
# Evaluated on the same dbn_test 2000 row sample used for DBN evaluation
# giving a direct comparison.

baseline_sample = dbn_test.sample(min(2000, len(dbn_test)), random_state=0).reset_index(drop=True)

print('=== Naive Persistence Baseline (predict today = yesterday) ===\n')
for target in DBN_TARGET_COLS:
    pred_base   = baseline_sample[f'{target}_prev'].astype(int)
    actual_base = baseline_sample[target].astype(int)
    print(f'--- {target} ---')
    print(classification_report(actual_base, pred_base, zero_division=0))


## DBN: Forecasting Atmospheric Variables

 **temperature, humidity, wind speed, and pressure**. All five atmospheric variables are modelled as nodes in the DBN, so we can issue MAP next day predictions for each. The persistence baseline (predict tomorrow = today) is put for context.

In [ ]:

# ── DBN: predict tomorrows discretized atmospheric variable classes ───────────

ATMO_TARGETS = ['TEMP_D', 'SLP_D', 'WIND_D', 'DEWP_D', 'PRCP_D']
dbn_atmo_preds = {t: [] for t in ATMO_TARGETS}

for _, row in dbn_eval.iterrows():
    evidence = {col: row[col] for col in DBN_EVIDENCE_COLS if pd.notna(row[col])}
    for target in ATMO_TARGETS:
        try:
            q = dbn_infer.map_query(variables=[target], evidence=evidence, show_progress=False)
            dbn_atmo_preds[target].append(q[target])
        except Exception:
            dbn_atmo_preds[target].append(np.nan)

print('=== DBN Next Day Atmospheric Variable Forecast ===')
print(f'{"Variable":<10}  {"DBN accuracy":>13}  {"Persistence":>11}  {"n":>6}')
print('-' * 48)
for target in ATMO_TARGETS:
    pred   = pd.Series(dbn_atmo_preds[target])
    actual = dbn_eval[target].reset_index(drop=True)
    prev   = dbn_eval[f'{target}_prev'].reset_index(drop=True)
    mask   = pred.notna() & actual.notna()
    acc      = (pred[mask] == actual[mask]).mean()
    base_acc = (prev[mask] == actual[mask]).mean()
    print(f'{target:<10}  {acc:>13.4f}  {base_acc:>11.4f}  {mask.sum():>6}')


### Probabilistic Evaluation: Brier Score

Accuracy and F1 only evaluate MAP predictions. Since the BN natively outputs posterior probability distributions, we measure the **Brier score:** a proper scoring rule (lower = better, 0 = perfect). The **Brier Skill Score** normalises against a baseline that always predicts the event frequency so values above 0 indicate the model adds real forecast skill

In [ ]:

# ── Probabilistic Evaluation Brier Score ─────────────────────────────────────
# Accuracy and F1 only assess MAP predictions.  Because the model natively
# outputs posterior distributions we can compute the Brier score
from sklearn.metrics import brier_score_loss

brier_rows = []
for target in DBN_TARGET_COLS:
    probs   = []
    actuals = []
    for _, row in dbn_eval.iterrows():
        evidence = {col: row[col] for col in DBN_EVIDENCE_COLS if pd.notna(row[col])}
        try:
            result = dbn_infer.query(variables=[target], evidence=evidence, show_progress=False)
            states = list(result.state_names[target])
            idx_1  = states.index(1)
            probs.append(float(result.values[idx_1]))
            actuals.append(int(row[target]))
        except Exception:
            pass

    base_rate = float(np.mean(actuals))
    bs_dbn    = brier_score_loss(actuals, probs)
    bs_base   = brier_score_loss(actuals, [base_rate] * len(actuals))
    skill     = 1 - bs_dbn / bs_base if bs_base > 0 else float('nan')

    brier_rows.append({
        'target':         target,
        'DBN_brier':      round(bs_dbn,  4),
        'baseline_brier': round(bs_base, 4),
        'brier_skill':    round(skill,   4),   # >0 means DBN beats climatology
    })

brier_df = pd.DataFrame(brier_rows)
print('=== Brier Score: DBN Probabilistic Forecast Quality ===\n')
print(brier_df.to_string(index=False))
print('\nBrier Skill Score > 0 means the DBN beats the climatological baseline.')


### Latent State DBN via HMM Inferred Regimes

The earlier `REGIME` experiment used same day clustering labels as if they were hidden variables. That makes the regime mostly a relabeling of the observations, which hurts the idea that the model is learning an unobserved atmospheric driver.

- we fit a Gaussian Hidden Markov Model (HMM) on each stations continuous weather sequence (`TEMP`, `DEWP`, `WDSP`, `SLP`, `PRCP`) to infer a small number of persistent weather regimes to work with
- we use the decoded regime sequence only as a **training time proxy** for the latent variable
- In the DBN we make both `REGIME_prev` and `REGIME` hidden at forecast time. The model only sees previous days observed weather variables and must infer the regime distribution internally

In [ ]:
%pip install hmmlearn

In [ ]:
# ── Latent State DBN: infer atmospheric regimes with an HMM ────────────────────
import numpy as np
import pandas as pd
from hmmlearn.hmm import GaussianHMM
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.estimators import BayesianEstimator
from pgmpy.inference import VariableElimination

# Load raw weather data and keep the continuous fields for the HMM
latent_src = pd.read_csv('../data/processed/gsod_2002.csv', low_memory=False)
latent_src['DATE'] = pd.to_datetime(latent_src['DATE'])
latent_src['STN'] = latent_src['STN'].astype(str)
latent_src['PRCP'] = latent_src['PRCP'].fillna(0)

continuous_cols = ['TEMP', 'DEWP', 'WDSP', 'SLP', 'PRCP']
binary_cols = ['FOG', 'RAIN', 'SNOW', 'THUNDER']
latent_src = latent_src.dropna(subset=['TEMP', 'DEWP', 'WDSP', 'SLP']).copy()
latent_src = latent_src.sort_values(['STN', 'DATE']).reset_index(drop=True)

# Use a time ordered split so the HMM and DBN only learn from the past
date_sorted = latent_src.sort_values('DATE').reset_index(drop=True)
split_idx = int(len(date_sorted) * 0.8)
split_date = date_sorted.iloc[split_idx]['DATE']
train_mask = latent_src['DATE'] <= split_date

# Standardize using train data only to avoid leaking test information
scaler = StandardScaler()
scaler.fit(latent_src.loc[train_mask, continuous_cols])

train_sequences = []
train_lengths = []
for _, station_frame in latent_src.loc[train_mask].groupby('STN', sort=False):
    if len(station_frame) < 2:
        continue
    scaled_values = scaler.transform(station_frame[continuous_cols])
    train_sequences.append(scaled_values)
    train_lengths.append(len(station_frame))

if not train_sequences:
    raise ValueError('No station sequences were long enough to train the latent HMM')

n_regimes = 3
latent_hmm = GaussianHMM(
    n_components=n_regimes,
    covariance_type='diag',
    n_iter=200,
    random_state=0,
    min_covar=1e-3,
    init_params='stmc'
    )
latent_hmm.fit(np.vstack(train_sequences), lengths=train_lengths)

# Determine the most likely regime sequence for every station using the fitted HMM
latent_src['REGIME'] = ''
for _, station_frame in latent_src.groupby('STN', sort=False):
    scaled_values = scaler.transform(station_frame[continuous_cols])
    decoded_states = latent_hmm.predict(scaled_values)
    latent_src.loc[station_frame.index, 'REGIME'] = [f'regime_{state}' for state in decoded_states]

# Discretize the observed weather variables used by the DBN
cut_specs = {
    'TEMP_D': (latent_src['TEMP'],           [-np.inf, 32, 75, np.inf],   ['cold', 'avg', 'warm']),
    'SLP_D':  (latent_src['SLP'],            [-np.inf, 1005, np.inf],      ['low', 'high']),
    'WIND_D': (latent_src['WDSP'],           [-np.inf, 5, 15, np.inf],     ['calm', 'moderate', 'strong']),
    'DEWP_D': (latent_src['DEWP'],           [-np.inf, 32, 55, np.inf],    ['low', 'moderate', 'high']),
    'PRCP_D': (latent_src['PRCP'],           [-np.inf, 0.0, 0.5, np.inf],  ['none', 'light', 'heavy']),
}

disc_latent = latent_src[['STN', 'DATE', 'REGIME'] + binary_cols].copy()
for col, (series, bins, labels) in cut_specs.items():
    disc_latent[col] = pd.cut(series, bins=bins, labels=labels)

discrete_cols = list(cut_specs.keys())
disc_latent = disc_latent.dropna(subset=discrete_cols).copy()
for col in discrete_cols:
    disc_latent[col] = disc_latent[col].astype(str)

observed_cols = discrete_cols + binary_cols
shift_cols = observed_cols + ['REGIME']
disc_latent = disc_latent.sort_values(['STN', 'DATE']).reset_index(drop=True)
for col in shift_cols:
    disc_latent[f'{col}_prev'] = disc_latent.groupby('STN')[col].shift(1)

latent_df = disc_latent.dropna(subset=[f'{col}_prev' for col in shift_cols]).copy()
for col in binary_cols:
    latent_df[col] = latent_df[col].astype(int)
    latent_df[f'{col}_prev'] = latent_df[f'{col}_prev'].astype(int)

# Hidden state design:
# - REGIME_prev explains the previous day observations
# - REGIME_prev -> REGIME carries the latent state forward one day
# - REGIME explains the current atmospheric variables and weather events
# - Direct observed persistence/intra slice edges preserve the forecasting structure
inter_slice_edges = [
    ('TEMP_D_prev', 'TEMP_D'),
    ('SLP_D_prev', 'SLP_D'),
    ('WIND_D_prev', 'WIND_D'),
    ('DEWP_D_prev', 'DEWP_D'),
    ('PRCP_D_prev', 'PRCP_D'),
    ('RAIN_prev', 'RAIN'),
    ('SNOW_prev', 'SNOW'),
    ('FOG_prev', 'FOG'),
    ('THUNDER_prev', 'THUNDER'),
]

intra_slice_edges = [
    ('SLP_D', 'TEMP_D'),
    ('SLP_D', 'WIND_D'),
    ('SLP_D', 'PRCP_D'),
    ('TEMP_D', 'DEWP_D'),
    ('TEMP_D', 'PRCP_D'),
    ('TEMP_D', 'SNOW'),
    ('TEMP_D', 'THUNDER'),
    ('DEWP_D', 'FOG'),
    ('DEWP_D', 'RAIN'),
    ('DEWP_D', 'THUNDER'),
    ('PRCP_D', 'RAIN'),
    ('PRCP_D', 'SNOW'),
    ('WIND_D', 'FOG'),
]

latent_edges = [('REGIME_prev', 'REGIME')]
latent_edges += [('REGIME_prev', f'{col}_prev') for col in observed_cols]
latent_edges += [('REGIME', col) for col in observed_cols]

dbn_latent_model = DiscreteBayesianNetwork(
    inter_slice_edges + intra_slice_edges + latent_edges
)

latent_sorted = latent_df.sort_values('DATE').reset_index(drop=True)
latent_train = latent_sorted[latent_sorted['DATE'] <= split_date].drop(columns=['STN', 'DATE'])
latent_test = latent_sorted[latent_sorted['DATE'] > split_date].drop(columns=['STN', 'DATE'])

print(f'HMM regimes learned: {sorted(latent_df["REGIME"].unique())}')
print(f'Latent train rows: {len(latent_train):,} | Latent test rows: {len(latent_test):,}')

dbn_latent_model.fit(latent_train, estimator=BayesianEstimator, prior_type='K2')
dbn_latent_infer = VariableElimination(dbn_latent_model)

# Forecast using only previous day observed evidence
latent_evidence_cols = [f'{col}_prev' for col in observed_cols]
sample_row = latent_test.iloc[0]
sample_evidence = {col: sample_row[col] for col in latent_evidence_cols}

print('\nPrevious day evidence sample:')
print({key: sample_evidence[key] for key in latent_evidence_cols[:6]})

regime_prev_posterior = dbn_latent_infer.query(
    variables=['REGIME_prev'], evidence=sample_evidence, show_progress=False
)
regime_curr_posterior = dbn_latent_infer.query(
    variables=['REGIME'], evidence=sample_evidence, show_progress=False
)
rain_posterior = dbn_latent_infer.query(
    variables=['RAIN'], evidence=sample_evidence, show_progress=False
)

print('\nPosterior P(REGIME_prev | previous-day observations):')
print(regime_prev_posterior)
print('\nPosterior P(REGIME | previous-day observations):')
print(regime_curr_posterior)
print('\nPosterior P(RAIN tomorrow | previous-day observations):')
print(rain_posterior)

LATENT_TARGET_COLS = ['RAIN', 'SNOW', 'FOG', 'THUNDER']
# Exact inference is expensive on the full hold out set so evaluate on a fixed sample
latent_eval = latent_test.sample(min(500, len(latent_test)), random_state=0).reset_index(drop=True)
latent_preds = {target: [] for target in LATENT_TARGET_COLS}
for _, row in latent_eval.iterrows():
    evidence = {col: row[col] for col in latent_evidence_cols}
    for target in LATENT_TARGET_COLS:
        try:
            result = dbn_latent_infer.map_query(
                variables=[target],
                evidence=evidence,
                elimination_order='MinFill',
                show_progress=False,
            )
            latent_preds[target].append(int(result[target]))
        except Exception:
            latent_preds[target].append(np.nan)

latent_summary_rows = []
for target in LATENT_TARGET_COLS:
    pred = pd.Series(latent_preds[target], dtype='float')
    actual = latent_eval[target].astype(int).reset_index(drop=True)
    mask = pred.notna()

    y_true = actual[mask].astype(int)
    y_pred = pred[mask].astype(int)

    latent_summary_rows.append({
        'target': target,
        'rows_scored': int(mask.sum()),
        'accuracy': round(float(accuracy_score(y_true, y_pred)), 4),
        'macro_f1': round(float(f1_score(y_true, y_pred, average='macro', zero_division=0)), 4),
    })

print('\nLatent DBN holdout summary (500 rows as a sample):')
print(pd.DataFrame(latent_summary_rows))

#### Latent DBN Diagnostics

The table below expands the hold out evaluation with per target precision/recall/F1 and confusion matrices. This is still an approximation because the HMM decoded regimes are proxies, but the forecast now uses them as **hidden** variables at prediction time rather than as observed same day labels

In [ ]:
for target in LATENT_TARGET_COLS:
    pred = pd.Series(latent_preds[target], dtype='float')
    actual = latent_eval[target].astype(int).reset_index(drop=True)
    mask = pred.notna()

    y_true = actual[mask].astype(int)
    y_pred = pred[mask].astype(int)

    print(f'=== {target} (latent DBN, 500 row sample) ===')
    print(classification_report(y_true, y_pred, zero_division=0))

    cm = confusion_matrix(y_true, y_pred)
    cm_df = pd.DataFrame(
        cm,
        index=[f'true_{i}' for i in range(cm.shape[0])],
        columns=[f'pred_{i}' for i in range(cm.shape[1])],
    )
    print(cm_df)
    print()

### HMM Regime Characterization

Each HMM hidden state corresponds to a distinct large scale atmospheric pattern. By inverting the standardisation applied before training, we recover the physical meaning of each regime in original units revealing whether the model has learned

In [ ]:

# ── Interpret the HMM regimes in physical units ───────────────────────────────
# latent_hmm.means_ has shape (n_regimes, n_features): invert the StandardScaler
# to recover original meteorological units for interpretation

regime_means_physical = scaler.inverse_transform(latent_hmm.means_)
regime_chars = pd.DataFrame(
    regime_means_physical,
    columns=continuous_cols,          # TEMP, DEWP, WDSP, SLP, PRCP
    index=[f'regime_{i}' for i in range(n_regimes)]
).round(2)

print('=== HMM Regime Characteristics ===\n')
print(regime_chars.to_string())
print()

# Proportion of total training & test days assigned to each regime
regime_counts = latent_src['REGIME'].value_counts().sort_index()
regime_pcts   = (regime_counts / regime_counts.sum() * 100).round(1)
print('Regime day counts and prevalence:')
for r in range(n_regimes):
    key = f'regime_{r}'
    cnt = regime_counts.get(key, 0)
    pct = regime_pcts.get(key, 0.0)
    print(f'  {key}: {cnt:,} station days  ({pct}%)')


## Model Comparison Summary

macro F1 across the four binary weather events for every model and the persistence baseline. The Static BN is evaluated on a random split so the DBN and Latent DBN use a time ordered split to prevent look ahead bias. The persistence baseline uses the DBNs time ordered hold out for fairness

In [ ]:

# ── Macro F1 comparison for all models and the baseline ──────────
from sklearn.metrics import f1_score

SHARED_TARGETS = ['RAIN', 'SNOW', 'FOG', 'THUNDER']
comparison_rows = []

# 1: Persistence baseline
for target in SHARED_TARGETS:
    pred_b   = baseline_sample[f'{target}_prev'].astype(int)
    actual_b = baseline_sample[target].astype(int)
    comparison_rows.append({
        'model':    'Persistence Baseline',
        'target':   target,
        'macro_f1': round(f1_score(actual_b, pred_b, average='macro', zero_division=0), 4),
    })

# 2: Static BN
for target in TARGET_COLS:
    if target not in SHARED_TARGETS:
        continue
    pred_s   = pd.Series(predictions[target], dtype='float')
    actual_s = eval_df[target].astype(int).reset_index(drop=True)
    mask_s   = pred_s.notna()
    comparison_rows.append({
        'model':    'Static BN',
        'target':   target,
        'macro_f1': round(f1_score(actual_s[mask_s].astype(int),
                                   pred_s[mask_s].astype(int),
                                   average='macro', zero_division=0), 4),
    })

# 3: DBN
for target in DBN_TARGET_COLS:
    pred_d   = pd.Series(dbn_preds[target], dtype='float')
    actual_d = dbn_eval[target].astype(int).reset_index(drop=True)
    mask_d   = pred_d.notna()
    comparison_rows.append({
        'model':    'DBN',
        'target':   target,
        'macro_f1': round(f1_score(actual_d[mask_d].astype(int),
                                   pred_d[mask_d].astype(int),
                                   average='macro', zero_division=0), 4),
    })

# 4: Latent DBN
for target in LATENT_TARGET_COLS:
    pred_l   = pd.Series(latent_preds[target], dtype='float')
    actual_l = latent_eval[target].astype(int).reset_index(drop=True)
    mask_l   = pred_l.notna()
    comparison_rows.append({
        'model':    'Latent DBN',
        'target':   target,
        'macro_f1': round(f1_score(actual_l[mask_l].astype(int),
                                   pred_l[mask_l].astype(int),
                                   average='macro', zero_division=0), 4),
    })

comp_df = pd.DataFrame(comparison_rows)
pivot   = comp_df.pivot_table(index='model', columns='target', values='macro_f1', aggfunc='first')
model_order = ['Persistence Baseline', 'Static BN', 'DBN', 'Latent DBN']
pivot = pivot.reindex([m for m in model_order if m in pivot.index])

print('=== Macro F1 by Model and Target Where Higher is Better ===\n')
print(pivot.round(4).to_string())
print('\n Static BN uses a random split & DBN/Latent DBN use time ordered splits')


## Interactive Model

Use the cells below to choose a trained model, enter evidence, and test predicted probabilities.

- `static` expects same day observed variables such as `TEMP_D`, `SLP_D`, `WIND_D`, `DEWP_D`, `PRCP_D`.
- `dbn` expects previous day variables ending in `_prev`, such as `TEMP_D_prev` and `RAIN_prev`.
- `latent_dbn` also expects previous day observed variables ending in `_prev`, but keeps the latent regime hidden during inference.

In [ ]:
# Interactive forecast tester
available_models = {}

if 'infer' in globals():
    available_models['static'] = {
        'infer': infer,
        'allowed_evidence': ['SLP_D', 'TEMP_D', 'WIND_D', 'DEWP_D', 'PRCP_D'],
        'default_query': ['RAIN', 'SNOW', 'FOG', 'THUNDER'],
    }

if 'dbn_infer' in globals() and 'DBN_EVIDENCE_COLS' in globals():
    available_models['dbn'] = {
        'infer': dbn_infer,
        'allowed_evidence': DBN_EVIDENCE_COLS,
        'default_query': ['RAIN', 'SNOW', 'FOG', 'THUNDER', 'TEMP_D', 'SLP_D', 'WIND_D', 'DEWP_D', 'PRCP_D'],
    }

if 'dbn_latent_infer' in globals() and 'latent_evidence_cols' in globals():
    available_models['latent_dbn'] = {
        'infer': dbn_latent_infer,
        'allowed_evidence': latent_evidence_cols,
        'default_query': ['RAIN', 'SNOW', 'FOG', 'THUNDER', 'REGIME_prev', 'REGIME'],
    }

# Change these values to test your own scenario.
selected_model = 'dbn'
query_vars = ['RAIN', 'SNOW', 'FOG', 'THUNDER']
user_evidence = {
    'TEMP_D_prev': 'warm',
    'SLP_D_prev': 'high',
    'WIND_D_prev': 'moderate',
    'DEWP_D_prev': 'high',
    'PRCP_D_prev': 'light',
    'RAIN_prev': 1,
    'SNOW_prev': 1,
    'FOG_prev': 1,
    'THUNDER_prev': 1,
}

if not available_models:
    print('No trained models are loaded in memory yet.')
    print('Run the model-building cells above, then rerun this tester cell.')
elif selected_model not in available_models:
    print(f"Model '{selected_model}' is not currently loaded.")
    print(f'Available models: {sorted(available_models)}')
else:
    config = available_models[selected_model]
    allowed = set(config['allowed_evidence'])
    invalid_keys = sorted(set(user_evidence) - allowed)

    if invalid_keys:
        print(f'Invalid evidence for {selected_model}: {invalid_keys}')
        print(f'Allowed keys: {sorted(allowed)}')
    else:
        missing_query = [var for var in query_vars if var not in config['infer'].variables]
        if missing_query:
            print(f'Query variables not supported by {selected_model}: {missing_query}')
            print(f'Try from: {config["default_query"]}')
        else:
            print(f'Loaded models: {sorted(available_models)}')
            print(f'Model: {selected_model}')
            print(f'Evidence: {user_evidence}')
            print()

            for var in query_vars:
                result = config['infer'].query(
                    variables=[var],
                    evidence=user_evidence,
                    show_progress=False,
                )
                probs = {
                    str(state): round(float(prob), 4)
                    for state, prob in zip(result.state_names[var], result.values)
                }
                print(f'{var}: {probs}')